# Backtest

In [ ]:
from typing import Literal, Optional
from logging_system import setup_logging

from backtest.domain.entities.backtest_statistics import BacktestStats
from backtest.domain.services.backtest_validator import BacktestValidator
from backtest.domain.validations import (
    MinSolAmountValidation,
    MaxSolAmountValidation,
    TradeActivityValidation,
    AllowedPoolsValidation
)
from backtest.infrastructure.adapters.presenters import show_stats
from backtest.infrastructure.container import Container

In [ ]:
# Configurar logging (opcional)
setup_logging(
    console_output=True,
    file_output=False,
    min_level_to_process="ERROR",
    enable_logfire=False,
)

data_source: Optional[Literal["file", "database"]] = None

# Crear container
container = Container()

### Cargar datos desde fichero (Datos de moralis)

In [ ]:
# ============================================================================
# OPCIÓN 1: Cargar datos desde archivo (Moralis)
# ============================================================================
# Obtener repositorio Moralis
repository = container.get_repository('moralis')

# Cargar desde archivo
repository.load_from_file("./data_moralis/4BdKax-buy-sell-2025-12-01-2025-12-22.json")
data_source = "file"

### Cargar desde Base de datos (Datos del trader)

In [ ]:
# ============================================================================
# OPCIÓN 2: Cargar datos desde Base de datos (PumpPortal o System)
# ============================================================================
# Obtener repositorio PumpPortal
repository = container.get_repository('pumpportal')
repository.load_from_database(
    system_wallet_address="48FJ2eB9ug7pqUZHSpNpL9oSvoPYRFtYEFeH3Jkty7dG",
    trader_wallet="CyaE1VxvBrahnPWkqm5VsdCvyS2QmNht2UFrKJHga54o",
    run_id=None,
    start_date=None,
    end_date=None,
    limit=None
)
data_source = "database"

In [ ]:
# ============================================================================
# Configurar validador (opcional)
# ============================================================================
validator = BacktestValidator([
    {
        "validations": [
            {
                "validations": [
                    MinSolAmountValidation(
                        exchange_thresholds={
                            "pump_swap": "3",
                            "pump_fun": "3"
                        },
                        enabled=True
                    ),
                    MaxSolAmountValidation(
                        exchange_thresholds={
                            "pump_swap": "4",
                            "pump_fun": "4"
                        },
                        enabled=False
                    )
                ],
                "logical_operator": "AND"
            },
            TradeActivityValidation(
                min_trade_count_threshold=3,
                activity_window_seconds=60,
                enabled=False
            )
        ],
        "logical_operator": "OR"
    },
    AllowedPoolsValidation(
        allowed_pools=["pump_swap", "pump_fun"],
        enabled=True
    )
])


In [ ]:
# ============================================================================
# Ejecutar backtest usando el container
# ============================================================================
if data_source is None:
    raise ValueError("data_source is None, please load data from 'file' or 'database'")

# Determinar el tipo de repositorio usado para obtener el runner correcto
from backtest.infrastructure.adapters.repositories import (
    MoralisTransactionRepository,
    PumpPortalTransactionRepository,
    SystemTransactionRepository
)

if isinstance(repository, MoralisTransactionRepository):
    repository_source = 'moralis'
elif isinstance(repository, PumpPortalTransactionRepository):
    repository_source = 'pumpportal'
elif isinstance(repository, SystemTransactionRepository):
    repository_source = 'system'
else:
    repository_source = 'moralis'  # Default

# Obtener el runner con todas las dependencias inyectadas
runner = container.get_backtest_runner(repository_source)

# Ejecutar backtest
# Puedes pasar validator=None para no aplicar validaciones
stats: Optional[BacktestStats] = runner.run(
    validator=validator,  # O None para no aplicar validaciones
    start_date=None,      # O "2025-01-01" o datetime object
    end_date=None         # O "2025-12-31" o datetime object
)

### Resultados del backtest

In [ ]:
if stats is not None:
    show_stats(stats)
else:
    print("No se encontraron estadísticas para mostrar")

## Comparación Trader vs Sistema

Compara las estadísticas del trader (repositorio pumpportal) con las del sistema (repositorio system), emparejando los trades por token_address y mostrando métricas avanzadas comparables.

In [ ]:
from logging_system import setup_logging

from backtest.domain.services.backtest_validator import BacktestValidator
from backtest.infrastructure.adapters.presenters import show_comparison
from backtest.infrastructure.container import Container

# Configurar logging
setup_logging(
    console_output=True,
    file_output=False,
    min_level_to_process="WARNING",
    enable_logfire=False,
)

# Crear container
container = Container()

### Paso 1: Cargar datos del TRADER (pumpportal)

In [ ]:
# Cargar repositorio del trader
trader_repository = container.get_repository('pumpportal')

# Opción 1: Cargar desde base de datos
trader_repository.load_from_database(
    system_wallet_address="48FJ2eB9ug7pqUZHSpNpL9oSvoPYRFtYEFeH3Jkty7dG",
    trader_wallet="CyaE1VxvBrahnPWkqm5VsdCvyS2QmNht2UFrKJHga54o",
    run_id=None,
    start_date=None,
    end_date=None,
    limit=None
)

# Ejecutar backtest del trader
trader_runner = container.get_backtest_runner('pumpportal')
trader_validator = BacktestValidator()  # Sin filtros, o agrega los que necesites

trader_stats = trader_runner.run(validator=trader_validator)

if trader_stats:
    print(f"✓ Backtest del trader: {trader_stats.total_trades} trades")
    print(f"  ROI: {float(trader_stats.roi):.2f}%")
    print(f"  Win Rate: {float(trader_stats.win_rate):.2f}%")
else:
    print("⚠️  No se encontraron trades para el trader")
    trader_stats = None

### Paso 2: Cargar datos del SISTEMA (system)

In [ ]:
# Cargar repositorio del sistema
system_repository = container.get_repository('system')

# Opción 1: Cargar desde base de datos
system_repository.load_from_database(
    system_wallet_address="48FJ2eB9ug7pqUZHSpNpL9oSvoPYRFtYEFeH3Jkty7dG",
    trader_wallet="CyaE1VxvBrahnPWkqm5VsdCvyS2QmNht2UFrKJHga54o",
    run_id=None,
    start_date=None,
    end_date=None,
    limit=None
)

# Ejecutar backtest del sistema
system_runner = container.get_backtest_runner('system')
system_validator = BacktestValidator()  # Mismos filtros que el trader

system_stats = system_runner.run(validator=system_validator)

if system_stats:
    print(f"✓ Backtest del sistema: {system_stats.total_trades} trades")
    print(f"  ROI: {float(system_stats.roi):.2f}%")
    print(f"  Win Rate: {float(system_stats.win_rate):.2f}%")
else:
    print("⚠️  No se encontraron trades para el sistema")
    system_stats = None

### Paso 3: Emparejar trades y comparar

In [ ]:
if trader_stats and system_stats:
    # Obtener el comparador
    comparator = container.get_backtest_comparator()

    # Emparejar trades por token_address
    print("Emparejando trades por token_address...")
    trader_matched, system_matched = comparator.compare_and_match_stats(
        trader_stats=trader_stats,
        system_stats=system_stats
    )

    print(f"✓ Emparejamiento completado:")
    print(f"  Trader: {len(trader_matched.closed_trades)} trades emparejados")
    print(f"  Sistema: {len(system_matched.closed_trades)} trades emparejados")

    # Mostrar comparación completa con métricas avanzadas
    show_comparison(
        trader_stats=trader_matched,
        system_stats=system_matched
    )
else:
    print("⚠️  No se pueden comparar: faltan datos del trader o del sistema")

## Backtest Optimizer

In [ ]:
from logging_system import setup_logging
from backtest.application.helpers.validator_builders import(
    build_min_sol_amount_validator,
    build_in_range_sol_amount_validator,
    build_outside_range_sol_amount_validator,
    build_trade_activity_validator
)
from backtest.application.helpers.param_generators import (
    generate_min_sol_amount_params,
    generate_range_sol_amount_params,
    generate_trade_activity_params,
    generate_monthly_date_ranges
)
from backtest.infrastructure.adapters.presenters import show_optimizer_results
from backtest.infrastructure.container import Container

In [ ]:
setup_logging(
    console_output=True,
    file_output=False,
    min_level_to_process="ERROR",
    enable_logfire=False,
)

# 1. Crear container y cargar datos
container = Container()
repository = container.get_repository('moralis')
repository.load_from_file('./data_moralis/CyaE1V_buy-sell_2025-07-18_2025-12-23_PARTIAL.json')

# 2. Obtener optimizador (ya viene con BacktestRunner inyectado)
optimizer = container.get_backtest_optimizer('moralis')

In [ ]:
data = optimizer.search_from_configs(
    configs=[
        {
            "filter_type": "min_sol_amount",
            "filter_params": generate_min_sol_amount_params(start=0.5, end=7.0, step=0.5),
            "build_validator": build_min_sol_amount_validator
        },
        {
            "filter_type": "in_range_sol_amount",
            "filter_params": generate_range_sol_amount_params(start=1.0, end=7.0, step=1.0),
            "build_validator": build_in_range_sol_amount_validator
        },
        {
            "filter_type": "outside_range_sol_amount",
            "filter_params": generate_range_sol_amount_params(start=1.0, end=7.0, step=1.0),
            "build_validator": build_outside_range_sol_amount_validator
        },
        {
            "filter_type": "trade_activity",
            "filter_params": generate_trade_activity_params(
                min_thresholds=[2, 3, 4, 5],
                activity_windows=[30, 60, 120, 180, 240, 300]
            ),
            "build_validator": build_trade_activity_validator
        }
    ],
    #date_range=generate_monthly_date_ranges(year=2025)
)

In [ ]:
data = optimizer.data_sorted_by_pool_key(
    data=data,
    pool_name="pump_fun",
    key_in_pool="win_rate"
)

show_optimizer_results(
    data=data
)

## Backtest Optimizer Batch

In [ ]:
from typing import Dict, List, Tuple, Optional, Literal, cast
from collections import defaultdict
from logging_system import setup_logging
from backtest.application.use_cases import BacktestOptimizer
from backtest.application.types import ParameterSearchConfig, BacktestDataEntry, PoolData
from backtest.application.helpers.validator_builders import(
    build_min_sol_amount_validator,
    build_in_range_sol_amount_validator,
    build_outside_range_sol_amount_validator,
    build_trade_activity_validator
)
from backtest.application.helpers.param_generators import (
    generate_min_sol_amount_params,
    generate_range_sol_amount_params,
    generate_trade_activity_params,
    generate_monthly_date_ranges
)
from backtest.infrastructure.container import Container

setup_logging(
    console_output=True,
    file_output=False,
    min_level_to_process="ERROR",
    enable_logfire=False,
)

In [ ]:
class PoolDataExtended(PoolData):
    filter_type: str
    filter_params: List[str]
    date_start: Optional[str]
    date_end: Optional[str]


class BacktestOptimizerBatch:

    def __init__(self):
        self.container = Container()
        self.repository = self.container.get_repository('moralis')
        self.optimizer = self.container.get_backtest_optimizer('moralis')

    def run(self, trader_files: Dict[str, str], configs: List[ParameterSearchConfig], date_range: Optional[List[Tuple[str, str]]] = None):
        trader_data: Dict[str, List[BacktestDataEntry]] = {}
        for trader_address, trader_file in trader_files.items():
            self.repository.load_from_file(trader_file)
            data = self.optimizer.search_from_configs(
                configs=configs,
                date_range=date_range
            )
            trader_data[trader_address] = data

        return trader_data

    @staticmethod
    def filter_trader_data(trader_data: Dict[str, List[BacktestDataEntry]], num_trades_by_strategy: int):
        data_filtered: Dict[str, Dict[str, Dict[str, List[PoolDataExtended]]]] = {}
        for trader_address, data in trader_data.items():
            data_filtered[trader_address] = defaultdict[str, Dict[str, List[PoolDataExtended]]](lambda: defaultdict[str, List[PoolDataExtended]](list))
            metrics = ("sortino_ratio", "profit_factor", "gain_expectancy")
            for pool_name, metrics in [("pump_fun", metrics), ("pump_swap", metrics)]:
                for metric in metrics:
                    trade_data_by_strategy = BacktestOptimizer.data_sorted_by_pool_key(
                        data=data,
                        pool_name=cast(Literal["pump_fun", "pump_swap"], pool_name),
                        key_in_pool=metric
                    )

                    current_num_trades_saved = 0
                    for trade_data in trade_data_by_strategy:
                        if current_num_trades_saved >= num_trades_by_strategy:
                            break
                        if pool_name not in trade_data:
                            continue

                        trade_data_extended = trade_data[pool_name]
                        trade_data_extended["filter_type"] = trade_data["filter_type"]
                        trade_data_extended["filter_params"] = trade_data["filter_params"]
                        trade_data_extended["date_start"] = trade_data["date_start"]
                        trade_data_extended["date_end"] = trade_data["date_end"]

                        data_filtered[trader_address][pool_name][metric].append(trade_data_extended)
                        current_num_trades_saved += 1

        return data_filtered


In [ ]:
def show_optimizer_results(data: Dict[str, Dict[str, Dict[str, List[PoolDataExtended]]]]):
    """
    Muestra los resultados de optimización agrupados por trader, pool y métrica.
    
    Args:
        data: Diccionario anidado con estructura:
            {trader_address: {pool_name: {metric: [PoolDataExtended]}}}
    """
    if not data:
        print("No hay datos para mostrar")
        return

    total_traders = len(data)
    print("\n" + "=" * 100)
    print(f"RESULTADOS DE OPTIMIZACIÓN ({total_traders} trader{'s' if total_traders > 1 else ''})")
    print("=" * 100)

    for trader_address, pools_data in data.items():
        print(f"\n{'═' * 100}")
        print(f"TRADER: {trader_address}")
        print(f"{'═' * 100}")

        for pool_name in ["pump_fun", "pump_swap"]:
            if pool_name not in pools_data:
                continue

            pool_display_name = "🏊 Pump.Fun" if pool_name == "pump_fun" else "🏊 PumpSwap"
            print(f"\n{pool_display_name}")
            print(f"{'─' * 100}")

            metrics_data = pools_data[pool_name]
            for metric_name, pool_data_list in metrics_data.items():
                if not pool_data_list:
                    continue

                # Nombre de la métrica más legible
                metric_display = {
                    "sortino_ratio": "Sortino Ratio",
                    "profit_factor": "Profit Factor",
                    "gain_expectancy": "Gain Expectancy"
                }.get(metric_name, metric_name.replace("_", " ").title())

                print(f"\n📊 Ordenado por: {metric_display}")
                print(f"{'─' * 50}")

                for idx, pool_data in enumerate(pool_data_list, 1):
                    print(f"\n  {'─' * 80}")
                    print(f"  ESTRATEGIA #{idx}")
                    print(f"  {'─' * 80}")

                    # Información básica de la estrategia (PoolDataExtended)
                    if 'filter_type' in pool_data:
                        print(f"  Tipo de filtro: {pool_data['filter_type']}")
                    if 'filter_params' in pool_data:
                        filter_params = pool_data['filter_params']
                        if isinstance(filter_params, list):
                            print(f"  Parámetros: {', '.join(str(p) for p in filter_params)}")
                        else:
                            print(f"  Parámetros: {filter_params}")

                    if pool_data.get('date_start') or pool_data.get('date_end'):
                        date_start = pool_data.get('date_start', 'N/A')
                        date_end = pool_data.get('date_end', 'N/A')
                        print(f"  Rango de fechas: {date_start} a {date_end}")

                    # Métricas generales
                    print(f"\n  📊 ESTADÍSTICAS GENERALES")
                    print(f"    Total trades: {pool_data.get('total_trades', 0)}")
                    if 'win_rate' in pool_data:
                        print(f"    Win rate: {float(pool_data['win_rate']):.2f}%")
                    if 'roi' in pool_data:
                        print(f"    ROI: {float(pool_data['roi']):.2f}%")

                    # Métricas avanzadas
                    has_advanced_metrics = any(
                        key in pool_data for key in [
                            'profit_factor', 'gain_expectancy', 'gain_expectancy_adjusted',
                            'sharpe_ratio', 'sortino_ratio'
                        ]
                    )
                    if has_advanced_metrics:
                        print(f"\n  📈 MÉTRICAS AVANZADAS")
                        if 'profit_factor' in pool_data:
                            print(f"    Profit Factor: {float(pool_data['profit_factor']):.4f}")
                        if 'gain_expectancy' in pool_data:
                            print(f"    Gain Expectancy: {float(pool_data['gain_expectancy']):.6f}")
                        if 'gain_expectancy_adjusted' in pool_data:
                            print(f"    Gain Expectancy Adjusted: {float(pool_data['gain_expectancy_adjusted']):.6f}")
                        if 'sharpe_ratio' in pool_data:
                            print(f"    Sharpe Ratio: {float(pool_data['sharpe_ratio']):.4f}")
                        if 'sortino_ratio' in pool_data:
                            print(f"    Sortino Ratio: {float(pool_data['sortino_ratio']):.4f}")

                    # Análisis financiero
                    print(f"\n  💰 ANÁLISIS FINANCIERO")
                    if 'total_sol_invested' in pool_data:
                        print(f"    Total SOL invertido: {float(pool_data['total_sol_invested']):.6f} SOL")
                    if 'total_sol_recovered' in pool_data:
                        print(f"    Total SOL recuperado: {float(pool_data['total_sol_recovered']):.6f} SOL")
                    if 'net_profit_sol' in pool_data:
                        profit_sign = "+" if pool_data['net_profit_sol'] >= 0 else ""
                        print(f"    Beneficio neto: {profit_sign}{float(pool_data['net_profit_sol']):.6f} SOL")

    print(f"\n{'=' * 100}\n")

### Ejecuar las funciones

In [ ]:
backtest_optimizer_batch = BacktestOptimizerBatch()

data = backtest_optimizer_batch.run(
    trader_files={
        #"CyaE1VxvBrahnPWkqm5VsdCvyS2QmNht2UFrKJHga54o": "./data_moralis/CyaE1V_buy-sell_2025-07-18_2025-12-23_PARTIAL.json",
        #"Ez2jp3rwXUbaTx7XwiHGaWVgTPFdzJoSg8TopqbxfaJN": "./data_moralis/Ez2jp3_buy-sell_2025-04-25_2025-12-26_ALL.json",
        "4BdKaxN8G6ka4GYtQQWk4G4dZRUTX2vQH9GcXdBREFUk": "./data_moralis/4BdKax_buy-sell_2024-09-01_2025-12-30_ALL.json"
    },
    configs=[
        {
            "filter_type": "min_sol_amount",
            "filter_params": generate_min_sol_amount_params(start=0.5, end=7.0, step=0.5),
            "build_validator": build_min_sol_amount_validator
        },
        {
            "filter_type": "in_range_sol_amount",
            "filter_params": generate_range_sol_amount_params(start=1.0, end=7.0, step=1.0),
            "build_validator": build_in_range_sol_amount_validator
        },
        {
            "filter_type": "outside_range_sol_amount",
            "filter_params": generate_range_sol_amount_params(start=1.0, end=7.0, step=1.0),
            "build_validator": build_outside_range_sol_amount_validator
        },
        {
            "filter_type": "trade_activity",
            "filter_params": generate_trade_activity_params(
                min_thresholds=[2, 3, 4, 5],
                activity_windows=[30, 60, 120, 180, 240, 300]
            ),
            "build_validator": build_trade_activity_validator
        }
    ],
    #date_range=generate_monthly_date_ranges(year=2025)
)

data_filtered = backtest_optimizer_batch.filter_trader_data(
    trader_data=data,
    num_trades_by_strategy=10
)

show_optimizer_results(data_filtered)

In [ ]:
data_filtered